In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, gamma
import pandas as pd

# Lab 11: Approximate Bayesian Computation

The goal of this session is to explore several methods in Approximate Bayesian Computation. We will cover two applications, already discussed during the lecture: inference of the mean of a normal distribution and inference of a parameter of the Lotka-Volterra population model. 

## Part 1: The models

### Model 1: Normal distribution

We consider a random variable $X$ following a normal distribution $\mathcal{N}(\mu, \sigma^2)$ with known $\sigma^2 = 1$. We assume a normal prior over $\mu$: $\mu \sim \mathcal{N}(m_0, s_0^2)$ with $m_0 = 2$ and $s_0^2 = 0.5$. 

#### Question 1 (*)
We observe $\mathbf{x} = (x_1, \dotsc, x_N)$. Show that the posterior $p(\mu | \mathbf{x}, \sigma^2)$ is a normal distribution. You don't need to specify the mean and the value. 

The posterior parameters are computed in the following function:

In [ ]:
def normal_posterior(X, m0, s0, sigma):
    N = len(X)
    x_mean = np.mean(X)
    
    m  = (sigma**2 * m0 + s0**2 * N * x_mean)/(sigma**2 + N * s0**2)
    s2 = sigma * s0 / np.sqrt(N * s0**2 + sigma**2)
    return m, s2

### Model 2: Lotka-Volterra

The Lotka-Volterra model is a model of population dynamics describing a biological environment where two species interact: prey and predators. The prey population grows naturally, while the predators need to eat prey. This is summed up in the following stochastic system:
$$
\frac{dx_1}{dt} = \alpha x_1 - \beta x_1 x_2 + \varepsilon_1(t)
$$
and 
$$
\frac{dx_2}{dt} = - \gamma x_2 + \delta x_1 x_2 + \varepsilon_2(t)
$$
where $\varepsilon_1(t) \sim \mathcal{N}(0, \sigma^2)$ and $\varepsilon_2(t) \sim \mathcal{N}(0, \sigma^2)$. The model is a probabilistic model with parameters $(\alpha, \beta, \gamma, \delta, \sigma^2)$. 

An Euler integration of the Lotka-Volterra equations is proposed in the following methods:

In [ ]:
def lotka_volterra(prey, predator, a, b, c, d, dt):
    """
    Simulates the Lotka-Volterra model for one time step.

    Parameters:
        prey (float): Current prey population
        predator (float): Current predator population
        a (float): Prey growth rate
        b (float): Predation rate
        c (float): Predator death rate
        d (float): Predator reproduction rate
        dt (float): Time step size

    Returns:
        (float, float): Updated prey and predator populations
    """
    d_prey = (a * prey - b * prey * predator) * dt
    d_predator = (d * prey * predator - c * predator) * dt
    
    prey += d_prey
    predator += d_predator
    
    return prey, predator

def simulate_lotka_volterra(prey0, predator0, a, b, c, d, dt, t_max, noise_std=0.0):
    """
    Simulates the Lotka-Volterra model over time.

    Parameters:
        prey0 (float): Initial prey population
        predator0 (float): Initial predator population
        a (float): Prey growth rate
        b (float): Predation rate
        c (float): Predator death rate
        d (float): Predator reproduction rate
        dt (float): Time step size
        t_max (float): Maximum simulation time
        noise_std (float): Standard deviation of Gaussian noise to add to populations

    Returns:
        np.ndarray, np.ndarray, np.ndarray: Time, prey populations, predator populations
    """
    time = np.arange(0, t_max, dt)
    prey_pop = np.zeros_like(time)
    predator_pop = np.zeros_like(time)

    # Initial conditions
    prey = prey0
    predator = predator0

    for i, t in enumerate(time):
        prey_pop[i] = prey
        predator_pop[i] = predator
        prey, predator = lotka_volterra(prey, predator, a, b, c, d, dt)

        # Add Gaussian noise
        prey += np.random.normal(0, noise_std)
        predator += np.random.normal(0, noise_std)

        # Ensure populations are non-negative
        prey = max(prey, 0)
        predator = max(predator, 0)

    return time, prey_pop, predator_pop

This code can be used to simulate the population dynamics:

In [ ]:
# Lotka-Volterra model parameters
a = 0.1  # Prey growth rate
b = 0.02  # Predation rate
c = 0.1  # Predator death rate
d = 0.01  # Predator reproduction rate

# Simulation parameters
prey0 = 40          # Initial prey population
predator0 = 9       # Initial predator population
dt = 0.01           # Time step size
t_max = 100         # Maximum simulation time
noise_std = 0.5     # Standard deviation of Gaussian noise

# Run simulation
time, prey_pop, predator_pop = simulate_lotka_volterra(prey0, predator0, a, b, c, d, dt, t_max, noise_std)

# Plot results
plt.figure(figsize=(12, 6))
plt.plot(time, prey_pop, label='Prey Population', color='blue')
plt.plot(time, predator_pop, label='Predator Population', color='red')
plt.title('Lotka-Volterra Predator-Prey Model with Noise')
plt.xlabel('Time')
plt.ylabel('Population')
plt.legend()
plt.grid()
plt.show()

## Part 2: The rejection algorithm

We will now propose an implementation of the rejection algorithm. We remind that the rejection algorithm aims to sample from the posterior $p(\theta | \tilde{\mathbf{x}})$ and consists in iteratively:

1. Sampling a parameter $\theta$ from the prior
2. Sampling an observation $\mathbf{x}$ from the likelihood $p(\cdot \mid \theta)$
3. Accepting $\mathbf{x}$ if $d(S(\mathbf{x}), S(\tilde{\mathbf{x}})) < \varepsilon$

where $d(\cdot,\cdot)$ is a distance, $S(\cdot)$ a summary statistics and $\varepsilon > 0$ a parameter.

The general algorithm is implemented in the following function:

In [ ]:
def rejection_algorithm(prior_sample_func, likelihood_sample_func, dist_func, summary_stat_func, X_obs, eps, n_steps_ABC, N):
    """
    Rejection algorithm

    Parameters:
        prior_sampl_func (function): Function to sample from the prior
        likelihood_sample_func (function): Function to sample from the likelihood, given parameter theta
        dist_func (function): Function computing the distance between two vectors
        summary_stat_func (function): Function computing a summary statistics of an observation X
        X_obs (np.ndarray): Observed data
        eps (float): Acceptation threshold (> 0)
        n_steps_ABC (int): Number of steps to run
        N: Size of the dataset to sample at each step

    Returns:
        np.ndarray: Sample from the posterior using rejection algorithm
        float: Acceptation rate of the rejection algorithm
    """
    posterior_sample = []
    for t in range(n_steps_ABC):
        if t % 100 == 0: print(f'Iteration {t}/{n_steps_ABC}')
        theta = prior_sample_func()
        X = likelihood_sample_func(theta, N)
        D = dist_func(summary_stat_func(X), summary_stat_func(X_obs))
        if D < eps: posterior_sample.append(theta)
    acceptation_rate = len(posterior_sample) / n_steps_ABC
    return np.array(posterior_sample), acceptation_rate

We propose two distances: L1 and L2 distances.

In [ ]:
def L1_distance(S1, S2):
    return np.sum(np.abs(S2 - S1))

def L2_distance(S1, S2):
    return np.sum((S2 - S1)**2)

### Normal model (*)
*Questions 2-4 together account for 1 bonus point.*

#### Question 2

Write a code for sampling from the prior and for sampling from the posterior given $\mu$. Remember here that $\sigma^2 = 1$. 

In [ ]:
def sample_normal_prior(m, s2):
    # Your code here
    return 0

def sample_normal_likelihood(mu):
    # Your code here
    return 0

m0 = 2
s0 = 0.5

normal_prior_sample_func = lambda: sample_normal_prior(m0, s0)
normal_likelihood_sample_func = lambda mu, N: sample_normal_likelihood(mu)

#### Question 3

Propose and implement some summary statistics that you could use to describe a dataset.

In [ ]:
def S_normal_1(X):
    # Your code here
    return 0

def S_normal_2(X):
    # Your code here
    return 0

# Define as many as you want!

We can now put all pieces together and run the rejection algorithm. The results can be observed using a histogram plot, comparing it to the PDF of the true posterior (known analytically).

In [ ]:
def visualize_normal_posterior(posterior_sample, X_obs, m0, s0, sigma, bins=100, mu_min=0, mu_max=5):
    m, s = normal_posterior(X_obs, m0, s0, sigma)
    
    plt.figure()

    # Histogram of sampled mus
    plt.hist(posterior_sample, bins=bins, range=(mu_min, mu_max), color='blue', edgecolor='black', alpha=0.7, density=True, label='Samples')

    # True posterior PDF
    xs = np.linspace(mu_min, mu_max, 1000)
    pdf = norm.pdf(xs, loc=m, scale=s)
    plt.plot(xs, pdf, 'r-', lw=2, label='True posterior')

    # Adding labels and grid
    plt.title('Rejection algorithm for the normal')
    plt.xlabel('$\\mu$')
    plt.ylabel('Frequency')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.legend()

    plt.show()

#### Question 4

Run the rejection algorithm with the summary statistics and the distance of your choice. Visualize the results using the previous function. Compare different choices of statistics, and also with different values of epsilon. Observe the rejection rate. 

In [ ]:
N = 50
X_obs = 4 + np.random.randn(N)

eps = .1
n_steps_ABC = 1000

mu_sample, rate =  rejection_algorithm(normal_prior_sample_func, 
                                       normal_likelihood_sample_func, 
                                       L2_distance, # To be chosen
                                       S_normal_1, # To be chosen
                                       X_obs, 
                                       eps, # To be chosen
                                       n_steps_ABC, # To be chosen
                                       N)

visualize_normal_posterior(mu_sample, X_obs, m0, s0, 1, bins=100, mu_min=0, mu_max=5)

### Lotka-Volterra model (*)

*Questions 5-7 together account for 1 bonus point.*

For the Lotka-Volterra model, we will consider that $\beta, \gamma$ and $\delta$ are known and we will only infer $\alpha$.

We will now reproduce the same steps as for the normal model. The difference is that now the data will be time series, and not standard datasets. 

#### Question 5

Choose and implement a prior. Keep in mind that the parameter $\alpha$ must be non-negative.

In [ ]:
def sample_LV_prior(): # Add the parameters of the prior in argument
    # Your code here
    return 0

def sample_LV_likelihood(prey0, predator0, a, b, c, d, dt, t_max, noise_std):
    _, prey_pop, predator_pop = simulate_lotka_volterra(prey0, predator0, a, b, c, d, dt, t_max, noise_std)
    return (prey_pop, predator_pop)


LV_prior_sample_func = lambda: sample_LV_prior(m0, s0)

# Using the parameters of the LV model introduced above
# N is declared as argument for conformity with the code in the rejection algorithm, but is ignored (N = 1 here)
LV_likelihood_sample_func = lambda a, N: sample_LV_likelihood(prey0, predator0, a, b, c, d, dt, t_max, noise_std)

#### Question 6

A possibility summary statistics is the time series itself. It is computationally heavy to compare two time series though, and we would like to use other statistics. Propose and implement some. 

In [ ]:
def raw_series(X):
    (prey_pop, predator_pop) = X
    T = len(prey_pop)
    return np.array([prey_pop / T, predator_pop / T])

def summary_statistics_series_2(X):
    # Your code here
    return 0

The following function is used for the visualization of the sampled posterior:

In [ ]:
def visualize_LV_posterior(posterior_sample, bins=100, a_min=0, a_max=5):
    plt.figure()

    plt.hist(posterior_sample, bins=bins, range=(a_min, a_max), color='blue', edgecolor='black', alpha=0.7, density=True)

    # Adding labels and grid
    plt.title('Rejection algorithm for the Lotka-Volterra model')
    plt.xlabel('$\\alpha$')
    plt.ylabel('Frequency')
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    plt.show()

#### Question 7

Run the rejection algorithm with the summary statistics and the distance of your choice. Visualize the results using the previous function. Compare different choices of statistics, and also with different values of epsilon. Observe the rejection rate. You will notice that, unlike for the normal model, it is very difficult to conclude here on the quality of the results.

In [ ]:
# Your code here

## Part 3: ABC-MCMC

We have seen that the standard rejection algorithm is particularly inefficient, because it samples from regions of the parameter space that get often rejected. This forces to have a very large number of samples, only a very tiny proportion of which is actually accepted. 

ABC-MCMC is used to restrict the exploration of the parameter space to more interesting areas. The goal of this exercise is to implement a simple version of the ABC-MCMC algorithm. 

For this implementation, we will use a Gaussian kernel, i.e. $q(\theta^\prime, \theta) = \mathcal{N}(\theta^\prime; \theta, \Sigma)$.

#### Question 8 (*)

Implement a step of the ABC-MCMC algorithm and include it into the general ABC-MCMC procedure. Test your method on the normal and Lotka-Volterra models. Compare in particular the acceptance rates needed to reach similar results as with the rejection algorithm.

In [ ]:
# For ABC-MCMC, we need the prior PDF
LV_prior_pdf = lambda x: gamma.pdf(x, 1, scale=1.1)

def ABC_MCMC_step(current_theta, Sigma, prior_pdf, likelihood_sample_func, dist_func, summary_stat_func, X_obs, eps):
    # Sample theta' from N(current_theta, Sigma)
    proposed_theta = 0 # Your code here
    
    # Sample x' from theta'
    proposed_x = likelihood_sample_func(proposed_theta, 1)
    
    # Compute acceptance rate
    rate = 0 # Your code here
    rate = min(1, rate)
    
    return (np.random.rand() <= rate), proposed_theta

def ABC_MCMC(x0, theta0, T: int, Sigma, prior_sample_func, likelihood_sample_func, dist_func, summary_stat_func, X_obs, eps):
    pass  # Your code here